# lowpoly: проверка на internal test

Модель и порог уже выбраны на validation. Здесь ничего не обучаем и не подбираем.

## 1. Данные и прежнее разбиение

In [1]:
%pip install -q catboost iterative-stratification

from pathlib import Path
import hashlib
import json
import zipfile
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'data/train.csv').is_file()), None)
if ROOT is None:
    ROOT = Path('/content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main')
assert (ROOT / 'data/train.csv').is_file(), 'Не найдена папка проекта с data/train.csv'

package = ROOT / 'lowpoly_internal_test_assets.zip'
assert package.is_file(), f'Загрузи файл в папку проекта: {package}'
with zipfile.ZipFile(package) as archive:
    for name in archive.namelist():
        if not (ROOT / name).is_file():
            archive.extract(name, ROOT)

labels = pd.read_csv(ROOT / 'data/train.csv')
targets = labels.columns.drop('item_id').tolist()
split = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=.15, random_state=42)
train_val_index, test_index = next(split.split(labels, labels[targets]))
test = labels.iloc[test_index].reset_index(drop=True)
assert len(test) == 1344 and set(test.item_id).isdisjoint(labels.iloc[train_val_index].item_id)
print('Internal test:', len(test))

Mounted at /content/drive
Internal test: 1344


## 2. Сохранённые признаки

In [2]:
simple_cache = ROOT / 'labels/simple/models/dinov2'
manifest = json.loads((simple_cache / 'internal_test_embeddings.json').read_text(encoding='utf-8'))
assert manifest['model'] == 'dinov2_vits14'
assert manifest['item_ids'] == test.item_id.tolist()
assert manifest['labels_sha256'] == hashlib.sha256((ROOT / 'data/train.csv').read_bytes()).hexdigest()
views = np.load(simple_cache / 'internal_test_embeddings.npy', mmap_mode='r')
assert views.shape == (1344, 6, 384) and np.isfinite(views).all()

work = ROOT / 'labels/lowpoly'
config = json.loads((work / 'artifacts/experiment_config.json').read_text(encoding='utf-8'))
columns = config['experiments']['A_no_curvature']['columns']
base = pd.read_csv(ROOT / 'archive/artifacts/train_mesh_features_relative.csv').set_index('item_id')
extra = pd.read_csv(work / 'artifacts/geometry_test_features.csv').set_index('item_id')
assert extra.index.is_unique and set(extra.index) == set(test.item_id)
geometry = base.join(extra[[name for name in columns if name not in base]])
geometry = geometry.loc[test.item_id, columns].to_numpy(dtype=np.float32)
assert geometry.shape == (1344, 78) and not np.isinf(geometry).any()

visual = np.concatenate([views.mean(axis=1), views.max(axis=1)], axis=1)
combined = np.concatenate([visual, geometry], axis=1)
assert combined.shape == (1344, 846)
print('DINOv2:', visual.shape, '| геометрия:', geometry.shape)

DINOv2: (1344, 768) | геометрия: (1344, 78)


## 3. Предсказание с прежними порогами

In [3]:
model_dir = work / 'models/dinov2'
validation = pd.read_csv(model_dir / 'metrics.csv').set_index('Experiment')
threshold = float(validation.loc['D3_mean_max_geometry', 'Threshold'])
assert abs(threshold - .26) < 1e-8

model = CatBoostClassifier()
model.load_model(str(model_dir / 'D3_mean_max_geometry.cbm'))
assert len(model.feature_names_) == combined.shape[1]
probability = model.predict_proba(combined)[:, 1]

old_model = CatBoostClassifier()
old_model.load_model(str(work / 'models/A_no_curvature.cbm'))
old_threshold = float(config['experiments']['A_no_curvature']['threshold'])
old_probability = old_model.predict_proba(geometry)[:, 1]
print('Пороги зафиксированы до test:', old_threshold, 'и', threshold)

Пороги зафиксированы до test: 0.23000000000000004 и 0.26


## 4. Результат и ошибки

In [4]:
y = test.lowpoly.to_numpy()

def score(p, threshold):
    prediction = p >= threshold
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    return {'Precision': precision_score(y, prediction, zero_division=0),
            'Recall': recall_score(y, prediction, zero_division=0),
            'F1': f1_score(y, prediction, zero_division=0),
            'AP': average_precision_score(y, p),
            'FP': int(fp), 'FN': int(fn), 'Threshold': threshold}

results = pd.DataFrame([
    {'Model': 'Geometry', **score(old_probability, old_threshold)},
    {'Model': 'DINOv2 + geometry', **score(probability, threshold)}
])
display(results.round(4))

predictions = test[['item_id', 'lowpoly', 'simple', 'noisy']].copy()
predictions['geometry_probability'] = old_probability
predictions['dino_geometry_probability'] = probability
predictions['prediction'] = (probability >= threshold).astype(int)
for label in ['simple', 'noisy']:
    group = predictions.query(f'lowpoly == 1 and {label} == 1')
    print(f'lowpoly + {label}: {len(group)} объектов, {int((group.prediction == 0).sum())} пропусков')

out = work / 'models/dinov2'
results.to_csv(out / 'internal_test_metrics.csv', index=False)
predictions.to_csv(out / 'internal_test_probabilities.csv', index=False)
print('Сохранено в', out)

,Model,Precision,Recall,F1,AP,FP,FN,Threshold
0,Geometry,0.5312,0.6145,0.5698,0.5881,90,64,0.23
1,DINOv2 + geometry,0.6125,0.5904,0.6012,0.6423,62,68,0.26


lowpoly + simple: 20 объектов, 15 пропусков
lowpoly + noisy: 14 объектов, 12 пропусков
Сохранено в /content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main/labels/lowpoly/models/dinov2
